In [36]:
import pandas as pd
import numpy as np


In [37]:
from pathlib import Path

p = Path('data/processed/train_data.parquet')

if p.exists():
    df = pd.read_parquet(p)
else:
    alternatives = [
        Path('data/processed/train.parquet'),
        Path('data/train_data.parquet'),
        Path('../data/processed/train_data.parquet'),
    ]
    found = next((a for a in alternatives if a.exists()), None)
    if found:
        if found.suffix == '.parquet':
            df = pd.read_parquet(found)
            
    else:
        proc = list(Path('data/processed').iterdir()) if Path('data/processed').exists() else []
        data_root = list(Path('data').iterdir()) if Path('data').exists() else []
        raise FileNotFoundError(
            f"train_data.parquet not found at {p!s}. Checked alternatives: {[str(a) for a in alternatives]}. "
            f"Files in data/processed: {[str(x) for x in proc]}. Files in data: {[str(x) for x in data_root]}."
        )

In [ ]:
df.head()

,hours_streaming,hours_social,hours_messaging,hours_gaming,is_peak_hour_user,is_weekend,streaming_data_gb,social_data_gb,messaging_data_gb,gaming_data_gb,...,plan_type_encoded,network_type_encoded,device_Basic_Phone,device_Mid_Range,device_Premium_Smartphone,device_Tablet,day_of_week_sin,day_of_week_cos,month_sin,month_cos
0,1.27,6.00,0.15,0.00,1,0,2.28387,1.83908,0.00208,0.00000,...,0.0,3.0,0,0,1,0,-0.433884,-0.900969,1.000000,6.123234e-17
1,0.33,0.22,1.79,0.31,0,0,0.12846,0.03291,0.02319,0.01750,...,2.0,0.0,0,0,1,0,0.000000,1.000000,1.000000,6.123234e-17
2,0.54,1.57,0.39,0.10,0,0,0.37170,0.19520,0.00540,0.01295,...,4.0,0.0,0,1,0,0,-0.974928,-0.222521,0.866025,-5.000000e-01
3,0.24,1.31,1.03,0.65,0,1,0.20155,0.11429,0.01589,0.08245,...,3.0,0.0,0,1,0,0,-0.781831,0.623490,0.866025,-5.000000e-01
4,1.90,0.68,0.01,0.22,0,0,4.30271,0.19975,0.00012,0.00968,...,2.0,3.0,1,0,0,0,-0.433884,-0.900969,0.866025,-5.000000e-01


In [41]:
import sys
import os
sys.path.append(os.path.abspath('.'))

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

def build_pipeline():

    remove_cols = [
        'user_id','data_usage_category','total_usage_gb',
        'top_activity','day_of_week','hour',
        'churn_risk_score','arpu_zar','arpu_per_gb'
    ]

    # Ordinal columns
    ordinal_cols = ['age_group', 'plan_type', 'network_type']

    age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
    plan_order = ['Prepaid_Daily', 'Prepaid_Monthly', 'Postpaid_Basic',
                  'Postpaid_Premium', 'Postpaid_Unlimited']
    network_order = ['3G', '4G', '4G+', '5G']

    ordinal_encoder = OrdinalEncoder(
        categories=[age_order, plan_order, network_order]
    )

    # One-hot
    nominal_cols = ['device_type']
    onehot = OneHotEncoder(drop='first', handle_unknown='ignore')

    # Column transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('ord', ordinal_encoder, ordinal_cols),
            ('nom', onehot, nominal_cols)
        ],
        remainder='passthrough'
    )

    pipeline = Pipeline(steps=[
        ('drop_cols', DropColumns(remove_cols)),
        ('date_features', DateFeatures()),
        ('cyclical', CyclicalFeatures()),
        ('encoding', preprocessor)
    ])

    return pipeline

def build_pipeline():

    remove_cols = [
        'user_id','data_usage_category','total_usage_gb',
        'top_activity','day_of_week','hour',
        'churn_risk_score','arpu_zar','arpu_per_gb'
    ]

    # Ordinal columns
    ordinal_cols = ['age_group', 'plan_type', 'network_type']

    age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
    plan_order = ['Prepaid_Daily', 'Prepaid_Monthly', 'Postpaid_Basic',
                  'Postpaid_Premium', 'Postpaid_Unlimited']
    network_order = ['3G', '4G', '4G+', '5G']

    ordinal_encoder = OrdinalEncoder(
        categories=[age_order, plan_order, network_order]
    )

    # One-hot
    nominal_cols = ['device_type']
    onehot = OneHotEncoder(drop='first', handle_unknown='ignore')

    # Column transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('ord', ordinal_encoder, ordinal_cols),
            ('nom', onehot, nominal_cols)
        ],
        remainder='passthrough'
    )

    pipeline = Pipeline(steps=[
        ('drop_cols', DropColumns(remove_cols)),
        ('date_features', DateFeatures()),
        ('cyclical', CyclicalFeatures()),
        ('encoding', preprocessor)
    ])

    return pipeline

In [42]:
train_alternatives = [
    Path('data/processed/train.parquet'),
    Path('data/train_data.parquet'),
    Path('../data/processed/train_data.parquet'),
]

test_alternatives = [
    Path('data/processed/test.parquet'),
    Path('data/test_data.parquet'),
    Path('../data/processed/test_data.parquet'),
]

def resolve_parquet_path(paths, name):
    path = next((p for p in paths if p.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"{name} not found. Checked paths: {[str(p) for p in paths]}"
        )
    return path

train_path = resolve_parquet_path(train_alternatives, "train data")
test_path = resolve_parquet_path(test_alternatives, "test data")

X_train = pd.read_parquet(train_path)
X_test = pd.read_parquet(test_path)


In [43]:
pipeline = build_pipeline()
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

In [44]:
print(X_train_processed.shape)
print(X_test_processed.shape)



(8004, 22)
(2002, 22)


In [46]:
import joblib
Path('models').mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, 'models/preprocessing_pipeline.pkl')

['models/preprocessing_pipeline.pkl']